In [2]:
# Setup the Jupyter version of Dash
from jupyter_dash import JupyterDash

# Configure the necessary Python module imports for dashboard components
import dash_leaflet as dl
from dash import dcc, html
import plotly.express as px
from dash import dash_table
from dash.dependencies import Input, Output, State
import base64
JupyterDash.infer_jupyter_proxy_config()

# Configure OS routines
import os

# Configure the plotting routines
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


# CRUD Python module completed for Project One
from CRUD_Python_Module import AnimalShelter

###########################
# Data Manipulation / Model
###########################
username = "aacuser"
password = "lol"

# Connect to database via CRUD Module
db = AnimalShelter(username, password)

# class read method must support return of list object and accept projection json input
# sending the read method an empty document requests all documents be returned
df = pd.DataFrame.from_records(db.read({}))

# MongoDB v5+ is going to return the '_id' column and that is going to have an
# invlaid object type of 'ObjectID' - which will cause the data_table to crash - so we remove
# it in the dataframe here. The df.drop command allows us to drop the column. If we do not set
# inplace=True - it will reeturn a new dataframe that does not contain the dropped column(s)
df.drop(columns=['_id'],inplace=True)

# Keep the full column list so every filtered result is shown with the
# same columns, in the same order, as the unfiltered table.
COLUMNS = list(df.columns)


# Rescue type queries, built from the Rescue Type and Preferred Dog Breeds
# table in the Dashboard Specifications Document. Each one matches dogs of
# the preferred breeds and sex within the training age range, using
# age_upon_outcome_in_weeks as the specification recommends.
#
# The Austin data truncates "Doberman Pinscher" to "Doberman Pinsch", so both
# spellings are included to make sure those dogs are not missed.
RESCUE_QUERIES = {
    'Water': {
        'animal_type': 'Dog',
        'breed': {'$in': ['Labrador Retriever Mix',
                          'Chesapeake Bay Retriever',
                          'Newfoundland']},
        'sex_upon_outcome': 'Intact Female',
        'age_upon_outcome_in_weeks': {'$gte': 26, '$lte': 156},
    },
    'Mountain': {
        'animal_type': 'Dog',
        'breed': {'$in': ['German Shepherd',
                          'Alaskan Malamute',
                          'Old English Sheepdog',
                          'Siberian Husky',
                          'Rottweiler']},
        'sex_upon_outcome': 'Intact Male',
        'age_upon_outcome_in_weeks': {'$gte': 26, '$lte': 156},
    },
    'Disaster': {
        'animal_type': 'Dog',
        'breed': {'$in': ['Doberman Pinscher',
                          'Doberman Pinsch',
                          'German Shepherd',
                          'Golden Retriever',
                          'Bloodhound',
                          'Rottweiler']},
        'sex_upon_outcome': 'Intact Male',
        'age_upon_outcome_in_weeks': {'$gte': 20, '$lte': 300},
    },
}


def query_records(filter_type):
    """Run the query for a rescue type through the CRUD module.

    Reset, or any value without a matching query, returns every document,
    which restores the original unfiltered view.
    """
    query = RESCUE_QUERIES.get(filter_type, {})
    results = pd.DataFrame.from_records(db.read(query))
    if results.empty:
        return []
    results = results.drop(columns=['_id'], errors='ignore')
    # Reindex so the columns always match the table definition, even if a
    # filtered result happens to be missing a field.
    return results.reindex(columns=COLUMNS).to_dict('records')


#########################
# Dashboard Layout / View
#########################
app = JupyterDash(__name__)

# Grazioso Salvare logo, encoded so it can be embedded directly in the page
image_filename = 'Grazioso Salvare Logo.png'
encoded_image = base64.b64encode(open(image_filename, 'rb').read())

app.layout = html.Div([
    # Branding: the logo links to the client's home page, as the
    # specification requires, followed by the title and unique identifier
    html.Center([
        html.A(
            html.Img(src='data:image/png;base64,{}'.format(encoded_image.decode()),
                     style={'height': '120px'}),
            href='https://www.snhu.edu', target='_blank'),
        html.B(html.H1('SNHU CS-340 Dashboard')),
        html.H3('Dashboard by Nikolas George (NGEORGE90675311)'),
    ]),
    html.Hr(),

    # Interactive filtering options. Radio buttons allow exactly one rescue
    # type at a time, and Reset returns everything to the unfiltered view.
    html.Div([
        html.B('Filter by rescue type: '),
        dcc.RadioItems(
            id='filter-type',
            options=[
                {'label': 'Water Rescue', 'value': 'Water'},
                {'label': 'Mountain or Wilderness Rescue', 'value': 'Mountain'},
                {'label': 'Disaster or Individual Tracking', 'value': 'Disaster'},
                {'label': 'Reset', 'value': 'Reset'},
            ],
            value='Reset',
            inline=True,
            inputStyle={'margin-left': '15px', 'margin-right': '5px'},
        ),
    ]),
    html.Hr(),

    dash_table.DataTable(
        id='datatable-id',
        columns=[{"name": i, "id": i, "deletable": False, "selectable": True} for i in COLUMNS],
        data=df.to_dict('records'),
        # Same client-friendly features as the Module Six milestone:
        # filtering and sorting on every column, ten rows per page, and
        # single row selection, which the map callback relies on.
        editable=False,
        filter_action="native",
        sort_action="native",
        sort_mode="multi",
        column_selectable=False,
        row_selectable="single",
        row_deletable=False,
        selected_columns=[],
        selected_rows=[0],
        page_action="native",
        page_current=0,
        page_size=10,
        style_table={'overflowX': 'auto'},
        style_cell={'textAlign': 'left', 'minWidth': '80px'},
    ),
    html.Br(),
    html.Hr(),
    # This sets up the dashboard so that your chart and your geolocation chart are side-by-side
    html.Div(className='row',
             style={'display': 'flex'},
             children=[
                 html.Div(id='graph-id', className='col s12 m6', style={'flex': '1'}),
                 html.Div(id='map-id', className='col s12 m6', style={'flex': '1'}),
             ])
])

#############################################
# Interaction Between Components / Controller
#############################################

# When a rescue type is chosen, query MongoDB through the CRUD module and
# replace the table's data with the results. The selected row is reset to
# the first row, since the previously selected row number may not exist in
# the new, smaller result set.
@app.callback([Output('datatable-id', 'data'),
               Output('datatable-id', 'selected_rows'),
               Output('datatable-id', 'page_current')],
              [Input('filter-type', 'value')])
def update_dashboard(filter_type):
    return query_records(filter_type), [0], 0


# Pie chart of the breeds in the table's current view. With no filter there
# are far too many breeds for a readable pie, so only the ten most common
# are shown and the rest are grouped as Other.
@app.callback(
    Output('graph-id', "children"),
    [Input('datatable-id', "derived_virtual_data")])
def update_graphs(viewData):
    if not viewData:
        return [html.P('No animals match this filter.')]

    dff = pd.DataFrame.from_dict(viewData)
    counts = dff['breed'].value_counts()
    if len(counts) > 10:
        top = counts.iloc[:10]
        top['Other'] = counts.iloc[10:].sum()
        counts = top

    chart_data = counts.reset_index()
    chart_data.columns = ['breed', 'count']
    figure = px.pie(chart_data, names='breed', values='count',
                    title='Breeds in Current View')
    # The chart column is narrow, so the legend goes below the pie instead
    # of beside it, where it would cover the slices.
    figure.update_layout(height=500, margin=dict(l=10, r=10, t=50, b=10),
                         legend=dict(orientation='h', y=-0.1,
                                     yanchor='top', x=0.5, xanchor='center'))
    return [dcc.Graph(figure=figure)]


# This callback will highlight a cell on the data table when the user selects it
@app.callback(
    Output('datatable-id', 'style_data_conditional'),
    [Input('datatable-id', 'selected_columns')]
)
def update_styles(selected_columns):
    return [{
        'if': {'column_id': i},
        'background_color': '#D2F3FF'
    } for i in selected_columns]


# This callback will update the geo-location chart for the selected data entry
# derived_virtual_data will be the set of data available from the datatable in the form of
# a dictionary.
# derived_virtual_selected_rows will be the selected row(s) in the table in the form of
# a list. For this application, we are only permitting single row selection so there is only
# one value in the list.
@app.callback(
    Output('map-id', "children"),
    [Input('datatable-id', "derived_virtual_data"),
     Input('datatable-id', "derived_virtual_selected_rows")])
def update_map(viewData, index):
    if not viewData:
        return []

    dff = pd.DataFrame.from_dict(viewData)
    # Because we only allow single row selection, the list can be converted to a row index here
    row = index[0] if index else 0
    if row >= len(dff):
        row = 0

    # Columns are looked up by name rather than by position
    lat = dff.iloc[row]['location_lat']
    lon = dff.iloc[row]['location_long']
    breed = dff.iloc[row]['breed']
    name = dff.iloc[row]['name']

    # Austin TX is at [30.75,-97.48]
    return [
        dl.Map(style={'width': '100%', 'height': '500px'}, center=[30.75, -97.48], zoom=10, children=[
            dl.TileLayer(id="base-layer-id"),
            # Marker with tool tip and popup
            dl.Marker(position=[lat, lon], children=[
                dl.Tooltip(breed),
                dl.Popup([
                    html.H1("Animal Name"),
                    html.P(name)
                ])
            ])
        ])
    ]


# Run app and display result inline in the notebook, note, if you have previously run a prior app, the default port of 8050 may not be available, if so, try setting an alternate port.
app.run_server(mode='inline', height=2100)

